In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_openai import ChatOpenAI

In [3]:
import subprocess

chrome_path = r"C:\Program Files\Google\Chrome\Application\chrome.exe"

subprocess.Popen(
    [
        chrome_path,
        "https://www.google.com"
    ]
)

<Popen: returncode: None args: ['C:\\Program Files\\Google\\Chrome\\Applicat...>

In [4]:
from langchain.tools import tool
import subprocess

CHROME_PATH = r"C:\Program Files\Google\Chrome\Application\chrome.exe"

@tool
def open_chrome(url: str) -> str:
    """Open a URL in Google Chrome on the user's computer."""

    subprocess.Popen(
        [
            CHROME_PATH,
            url
        ]
    )

    return f"Opened {url} in Google Chrome."

In [5]:
open_chrome.invoke({
    "url": "https://www.python.org"
})

'Opened https://www.python.org in Google Chrome.'

In [6]:
llm = ChatOpenAI(
    model = "openai/gpt-oss-20b",
    api_key = os.getenv("GROK_API_KEY"),
    base_url = "https://api.groq.com/openai/v1",
)

In [7]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[open_chrome],
    system_prompt=(
        "You are a desktop assistant. "
        "Use the open_chrome tool when the user asks you to open a website."
    )
)

In [8]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Open python.org in Chrome."
        }
    ]
})

print(response["messages"][-1].content)

Got it! python.org is now open in Chrome.


In [9]:
from langchain_tavily import TavilySearch

search_tool = TavilySearch(max_results=3)

result = search_tool.invoke({
    "query": "Latest LangChain features"
})

print(result)

{'query': 'Latest LangChain features', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://docs.langchain.com/oss/python/releases/changelog', 'title': 'Changelog - Docs by LangChain', 'content': 'New integration packages for pluggable sandboxes: `langchain-modal`, `langchain-daytona`, and `langchain-runloop`. See sandboxes guide and example data analysis tutorial.\n Changes to conversation history summarization:\n  + Summarization now happens in the model node via `wrap_model_call` events. Due to this we retain the full message history in the graph state.\n  + More accurate token counting. [...] `SystemMessage` for `create_agent`: Support for passing `SystemMessage` instances directly to `create_agent`’s `system_prompt` parameter, enabling advanced features like cache control and structured content blocks.\n Model retry middleware: New middleware for automatically retrying failed model calls with configurable exponential backoff. [...] Model profiles

In [10]:
result["results"]

[{'url': 'https://docs.langchain.com/oss/python/releases/changelog',
  'title': 'Changelog - Docs by LangChain',
  'content': 'New integration packages for pluggable sandboxes: `langchain-modal`, `langchain-daytona`, and `langchain-runloop`. See sandboxes guide and example data analysis tutorial.\n Changes to conversation history summarization:\n  + Summarization now happens in the model node via `wrap_model_call` events. Due to this we retain the full message history in the graph state.\n  + More accurate token counting. [...] `SystemMessage` for `create_agent`: Support for passing `SystemMessage` instances directly to `create_agent`’s `system_prompt` parameter, enabling advanced features like cache control and structured content blocks.\n Model retry middleware: New middleware for automatically retrying failed model calls with configurable exponential backoff. [...] Model profiles: Chat models now expose supported features and capabilities through a `.profile` attribute. These data a

In [12]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[open_chrome, search_tool],
    system_prompt="""
    You are a helpful browser research assistant.

    - Use open_chrome when the user asks you to open a website.
    - Use the search tool when the user asks for information or research.
    - When searching, summarize the results and include source URLs.
    - Only open websites when the user requests it or when it is useful
      and appropriate for their request.
    """
)

In [16]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Search for the latest LangGraph features and summarize three findings."
        }
    ]
})

print(response["messages"][-1].content)

**Three key take‑aways from LangGraph’s most recent feature set**

| # | Feature | What it does | Source |
|---|---------|--------------|--------|
| 1 | **Declarative error handlers & Saga‑style compensation** | Nodes can now define `error_handler=` callbacks that run after all retry attempts fail.  This lets developers implement *Saga* patterns directly in the graph: a failed step can trigger a cleanup or rollback node that undoes earlier work.  The new `NodeError` type is passed to the handler, allowing fine‑grained recovery logic without wrapping every node in `try/except`. | ["Deep Dive into LangGraph 1.2", n1n.ai, June 2026](https://explore.n1n.ai/blog/langgraph-1-2-deep-dive-fault-tolerance-streaming-2026-06-01) |
| 2 | **Per‑node timeouts, graceful shutdown, and resumable streams** | • `add_node(..., timeout=…)` caps how long a single node can run, preventing one slow LLM call from stalling the whole graph.  <br>• `RunControl.request_drain()` lets a pod terminate *cooperatively*

In [17]:
for message in response["messages"]:
    for call in getattr(message, "tool_calls", []):
        print("Tool:", call["name"])
        print("Arguments:", call["args"])

Tool: tavily_search
Arguments: {'query': 'latest LangGraph features', 'search_depth': 'advanced'}
Tool: open_chrome
Arguments: {'url': 'https://www.langchain.com/blog/langgraph-release-week-recap'}
Tool: tavily_search
Arguments: {'query': 'LangGraph 1.2 features', 'search_depth': 'advanced'}


In [18]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Open python.org in Chrome."
        }
    ]
})

print(response["messages"][-1].content)

Got it! The Python homepage is now open in your Chrome browser. Let me know if you’d like any additional information or help with anything else.


In [19]:
for message in response["messages"]:
    for call in getattr(message, "tool_calls", []):
        print("Tool:", call["name"])
        print("Arguments:", call["args"])

Tool: open_chrome
Arguments: {'url': 'https://www.python.org'}
